# Convolution / Deconvolution
---
**Notebook 5 — 2026**
<div style="font-size:15px; padding:8px; margin:2px; font-weight:600; background-color:#E80808; color:white;text-align:center;">
    <div style=" ">
        Introduction to Multidimensional Fourier Transform
    </div>
</div>
<div style="border-bottom: 1px gray dotted; padding:8px;margin:2px;text-align:center; font-size:15px; color:#444;">
<i>Daniel Sage — École Polytechnique Fédérale de Lausanne (EPFL)</i>
</div>

**Important notice**: These interactive notebooks complement the lecture and are not self-contained; in-class explanations are required.

In [1]:
%pip install -q ipywidgets
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, transform, draw, io
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, Layout
from matplotlib.patches import Circle
layout = Layout(width='400px') # Layout widgets
style  = {'description_width': '200px'} # Style widgets
plt.rcParams["figure.dpi"]=120
plt.rcParams["image.cmap"] = "gray"

def fft2c(x): return np.fft.fftshift(np.fft.fft2(x))
def ifft2c(F): return np.real(np.fft.ifft2(np.fft.ifftshift(F)))
def snr(x,y): return 10*np.log10(np.mean(x*x)/(np.mean((x-y)**2)+1e-12))
def radial_grid(n): a=(np.arange(n)-n//2)/n; X,Y=np.meshgrid(a,a); return np.sqrt(X*X+Y*Y)
def showim(ax,img,t,vmin=None,vmax=None): ax.imshow(img,cmap="gray",vmin=vmin,vmax=vmax); ax.set_title(t); ax.axis("off")
def showFM(ax,F,t): ax.imshow(np.log(np.abs(F)+1e-12),cmap="gray"); ax.set_title(t); ax.axis("off")

size=128
img = transform.resize(data.camera(), (size, size), anti_aliasing=True)


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip3.8 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
def load(choice, n=128):
    sources = {"car_pad": lambda: io.imread("car_pad.tif", as_gray=True), "cell_substrat": 
               lambda: io.imread("cell_substrat.tif", as_gray=True),
               "camera":data.camera, "brick":data.brick, "grass": data.grass,}
    img = sources[choice]()
    return transform.resize(img, (n, n), anti_aliasing=True)

selector = Dropdown(options=["car_pad", "cell_substrat", "camera", "brick", "grass"], description="Image")
def update_img(change): global img; img = load(change["new"])

selector.observe(update_img, names="value")
display(selector)

Dropdown(description='Image', options=('car_pad', 'cell_substrat', 'camera', 'brick', 'grass'), value='car_pad…

## 5.1 Convolution / Multiplication

In [3]:
@interact(ker_name=Dropdown(options=["gaussian","disk","dx","dy"], value="gaussian"))
def conv_theorem(ker_name):
    I = img
    y,x = np.meshgrid(np.arange(size)-size//2, np.arange(size)-size//2, indexing="ij")
    if ker_name=="gaussian": sigma=1.5; K = np.exp(-np.sqrt(x*x + y*y)**2/(2*sigma**2))
    elif ker_name=="disk": sigma=3; K = (np.sqrt(x*x + y*y) <= sigma).astype(float)
    elif ker_name=="dx": K = np.zeros_like(I); K[size//2, size//2+1] = 1; K[size//2, size//2-1] = -1
    else: K = np.zeros_like(I); K[size//2+1, size//2] = 1; K[size//2-1, size//2] = -1

    K /= np.sum(np.abs(K)) + 1e-12
    FI = fft2c(I); FK = fft2c(np.fft.ifftshift(K))
    Fconv = FI * FK; Iconv = ifft2c(Fconv)
    Iref = ifft2c(fft2c(I) * fft2c(np.fft.ifftshift(K)))
    diff = np.linalg.norm(Iconv - Iref)

    fig,ax = plt.subplots(2,4,figsize=(16,8))
    showim(ax[0,0], I, "Image (I)");  showFM(ax[0,1], FI, "FFT(I)")
    showim(ax[0,2], K, "Kernel (K)"); showFM(ax[0,3], FK, "FFT(K)")
    showim(ax[1,0], Iconv, "Convolution(I,K)"); showFM(ax[1,1], Fconv, "FFT(I)·FFT(K)")
    showim(ax[1,2], Iref, "IFFT(FFT(I)·FFT(K))")
    showim(ax[1,3], Iconv - Iref, f"Diff ‖·‖={diff:.3e}")
    plt.show()

interactive(children=(Dropdown(description='ker_name', options=('gaussian', 'disk', 'dx', 'dy'), value='gaussi…

## 5.2 Convolution / Deconvolution

In [49]:
def deconvolve_wiener(im, psf, power_wiener_reg=-3):
    regul = np.power(10.0,power_wiener_reg)
    H = fft2c(psf)
    h = ifft2c(np.conj(H)/(np.abs(H)**2 + regul)); 
    h = h / h.sum()
    wif = np.real(ifft2c(fft2c(im) * fft2c(np.fft.fftshift(h))))
    return wif
    
@interact(
    freqcut=FloatSlider(min=0.002, max=0.25, step=0.001, value=0.15,readout_format='.3f',description='Frequendy cut',layout=layout,style=style),
    noise=FloatSlider(min=0,max=1,step=0.1,value=0.1,readout_format='.3f',layout=layout,style=style),
    power_reg=FloatSlider(min=-10,max=10,step=0.5,value=-3,description='Power Regul.', layout=layout,style=style)

)
def demo(freqcut, noise, power_reg):
    H=(radial_grid(size)<=freqcut).astype(float)
    K=np.abs(ifft2c(H))**2; K=np.fft.fftshift(K)/K.sum()
    blur = ifft2c(fft2c(img)*H)
    noisy = blur + noise*np.random.randn(*blur.shape)
    deconv = deconvolve_wiener(noisy, K,power_wiener_reg=power_reg)
    fig,ax=plt.subplots(2,4,figsize=(10,5))
    showim(ax[0,0],img,f"original \nSNR={snr(img,img):.2f} dB")
    showim(ax[0,1],blur,f"blur \nSNR={snr(img,blur):.2f} dB")
    showim(ax[0,2],noisy,f"blur + noise \nSNR={snr(img,noisy):.2f} dB")
    showim(ax[0,3],deconv,f"deconv \nSNR={snr(img,deconv):.2f} dB")
    showFM(ax[1,0],fft2c(img),""); showFM(ax[1,1],fft2c(blur),"")
    showFM(ax[1,2],fft2c(noisy),""); showFM(ax[1,3],fft2c(deconv),"")
    plt.show()

interactive(children=(FloatSlider(value=0.15, description='Frequendy cut', layout=Layout(width='400px'), max=0…